# 03. 제로샷 데모 (KOSPI) + ARIMA 비교
`python run.py download-models --sizes 5M 209M` 이후 실행.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.common import load_config
from src.data.process import load_processed
from src.eval.backtest import backtest_frame, make_origins
from src.forecast.base import get_forecaster
from src.targets.spec import load_series_specs
from src.viz.plots import fan_chart
ecfg = load_config('experiment'); specs = load_series_specs()
df = load_processed('KOSPI'); series, h, ctx = 'm20', 20, 1024
raw = df[series]
origins = make_origins(raw, ecfg['test_start'], None, 20, h, 2048)   # 데모는 origin 20일 간격
print(len(origins), 'origins')

In [ ]:
res = {}
for m in ['timestral-209M', 'arima', 'climatology']:
    fc = get_forecaster(m, ecfg)
    res[m] = backtest_frame(raw, specs[series], fc, h, ctx if fc.uses_context else None, origins, key='KOSPI:m20')
    fan_chart(res[m], f'{m} | KOSPI {series} h{h}')

In [ ]:
from src.eval.metrics import pinball, violation_rate
for m, r in res.items():
    print(f"{m:16s} q05 violation={violation_rate(r.y_true, r.q05):.3f}  pinball05={pinball(r.y_true, r.q05, 0.05):.5f}")